In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import networkx as nx
from shapely import geometry
from shapely.geometry import Point, LineString
import osmnx as ox
from shapely.ops import unary_union, linemerge, substring, nearest_points
from scipy.spatial import cKDTree
import re
from tqdm import tqdm
import overpy
import time

# Build public transport network

## Get osm public transport data

In [ ]:
def stitch_relation_ways(rel, tol=0.0003):
    coords = []
    lines = []

    for m in rel.members:
        if not isinstance(m, overpy.RelationWay):
            continue
        try:
            way = m.resolve()
        except Exception:
            continue

        way_coords = [(float(n.lon), float(n.lat)) for n in way.nodes]
        if len(way_coords) < 2:
            continue

        if coords:
            last_pt = Point(coords[-1])
            first_pt = Point(way_coords[0])
            last_pt_rev = Point(way_coords[-1])

            # Direction Correction: Make it more likely for the new segment to connect to the end of the current one
            if last_pt.distance(first_pt) > last_pt.distance(last_pt_rev):
                way_coords.reverse()

            # If it connects, go for it; if not, break it off and start a new segment
            if last_pt.distance(Point(way_coords[0])) < tol:
                coords.extend(way_coords[1:])
            else:
                if len(coords) > 1:
                    lines.append(LineString(coords))
                coords = way_coords
        else:
            coords = way_coords

    if len(coords) > 1:
        lines.append(LineString(coords))

    return lines

def greedy_stitch_lines(lines, tol_deg=0.0008):
    if not lines:
        return None
    if len(lines) == 1:
        return lines[0]

    used = [False] * len(lines)
    base = max(range(len(lines)), key=lambda i: lines[i].length)
    used[base] = True
    coords = list(lines[base].coords)

    def try_attach(seg):
        nonlocal coords
        a0 = Point(coords[0])
        a1 = Point(coords[-1])
        b0 = Point(seg.coords[0])
        b1 = Point(seg.coords[-1])

        d_end_start = a1.distance(b0)
        d_end_end = a1.distance(b1)
        d_start_start = a0.distance(b0)
        d_start_end = a0.distance(b1)

        best = min(
            (d_end_start, "end_start"),
            (d_end_end, "end_end"),
            (d_start_start, "start_start"),
            (d_start_end, "start_end"),
            key=lambda x: x[0]
        )

        if best[0] > tol_deg:
            return False

        mode = best[1]
        if mode == "end_start":
            coords.extend(list(seg.coords)[1:])
        elif mode == "end_end":
            coords.extend(list(seg.coords)[-2::-1])
        elif mode == "start_start":
            coords = list(seg.coords)[-1:0:-1] + coords
        else:  # "start_end"
            coords = list(seg.coords)[:-1] + coords

        return True

    changed = True
    while changed:
        changed = False
        for i, seg in enumerate(lines):
            if used[i]:
                continue
            if try_attach(seg):
                used[i] = True
                changed = True

    return LineString(coords) if len(coords) > 1 else None


def extract_ordered_stops_from_relation(rel):
    """Retrieve stop nodes in the order specified by `relation.members` (unsorted)"""
    ordered = []
    seen = set()
    for m in rel.members:
        if not isinstance(m, overpy.RelationNode):
            continue
        try:
            n = m.resolve()
        except Exception:
            continue
        sid = int(n.id)
        # Duplicate Removal: The same stop may appear multiple times in a relation (platform/stop_position); you can also filter more precisely by role.
        if sid in seen:
            continue
        seen.add(sid)
        ordered.append((sid, Point(float(n.lon), float(n.lat))))
    return ordered

def edges_from_relation(rel, route_type, tol_deg=0.0008):
    edges = []

    stitched_lines = stitch_relation_ways(rel)
    if not stitched_lines:
        return edges

    u = unary_union(stitched_lines)
    if u.is_empty:
        return edges

    if u.geom_type == "LineString":
        merged = u
    elif u.geom_type == "MultiLineString":
        merged = linemerge(u)
    elif u.geom_type == "GeometryCollection":
        geoms = [g for g in u.geoms if g.geom_type in ("LineString", "MultiLineString")]
        if not geoms:
            return edges
        u2 = unary_union(geoms)
        merged = u2 if u2.geom_type == "LineString" else linemerge(u2)
    else:
        return edges

    if merged.geom_type == "LineString":
        route_line = merged
    else:
        route_line = greedy_stitch_lines(list(merged.geoms), tol_deg=tol_deg)
        if route_line is None:
            return edges

    # Use the "stop" order of the relation (without sorting)
    ordered_stops = extract_ordered_stops_from_relation(rel)
    if len(ordered_stops) < 2:
        return edges

    # Calculate the projection position (the “cumulative length” along route_line, measured in degrees)
    projs = []
    for sid, pt in ordered_stops:
        projs.append((sid, route_line.project(pt), pt))

    # Check whether the direction of `route_line` matches the order of `stops`: If the route is generally descending, reverse the line.
    # Conduct a poll based on the signs of adjacent differences
    diffs = [projs[i+1][1] - projs[i][1] for i in range(len(projs)-1)]
    vote = sum(1 if d > 0 else -1 for d in diffs if abs(d) > 1e-12)
    if vote < 0:
        route_line = LineString(list(route_line.coords)[::-1])
        projs = [(sid, route_line.project(pt), pt) for sid, _, pt in projs]

    # Now generate edges in relation order (unsorted)
    for i in range(len(projs) - 1):
        u_id, u_proj, _ = projs[i]
        v_id, v_proj, _ = projs[i + 1]

        if abs(v_proj - u_proj) < 1e-12:
            continue

        # The `substring` function requires `start <= end`, but we need to preserve the direction.
        if v_proj > u_proj:
            seg = substring(route_line, u_proj, v_proj)
        else:
            seg = substring(route_line, v_proj, u_proj)
            # The direction is reversed
            if seg.geom_type == "LineString":
                seg = LineString(list(seg.coords)[::-1])

        if seg.geom_type != "LineString" or len(seg.coords) < 2:
            continue

        edges.append({
            "from_stop": int(u_id),
            "to_stop": int(v_id),
            "route_id": int(rel.id),
            "route_name": rel.tags.get("name", ""),
            "route_type": route_type,
            "geometry": seg
        })

    return edges


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def safe_query(api, query, retries=6, base_sleep=3):
    last = None
    for k in range(retries):
        try:
            return api.query(query)
        except Exception as e:
            last = e
            time.sleep(base_sleep * (2 ** k))
    raise last

In [ ]:
# route_type: ["bus","tram","subway","light_rail","train"]
api = overpy.Overpass()

city = "newyork"
unit = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_tract.shp")
boundary = gpd.GeoDataFrame(geometry=[unit.union_all()], crs="EPSG:4326")
bbox = boundary.total_bounds

route_type = "train"

query_ids = f"""
[out:json][timeout:300];
relation["route"="{route_type}"]({bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]});
out ids;
"""
res_ids = api.query(query_ids)
rel_ids = sorted({int(r.id) for r in res_ids.relations})
print(f"{route_type} relation id 数量: {len(rel_ids)}")

edges_all = []
tol_deg = 0.0008

batch_size = 20  
for batch in chunks(rel_ids, batch_size):
    ids_str = ",".join(map(str, batch))
    query_full = f"""
    [out:json][timeout:300];
    relation(id:{ids_str});
    out body;
    >;
    out skel qt;
    """
    res = safe_query(api, query_full)

    for rel in res.relations:
        edges_all.extend(edges_from_relation(rel, route_type=route_type, tol_deg=tol_deg))

edges_gdf = gpd.GeoDataFrame(edges_all, crs="EPSG:4326")
print(f"{route_type} edges 数: {len(edges_gdf)}")
edges_gdf.to_file(f"D:/urban_hierarchy_congestion/data/transport_network/spatial_data/{city}/{route_type}_edges.shp")

## Build public transport network

In [ ]:
def normalize_stop_id(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    try:
        xf = float(s)
        if xf.is_integer():
            return str(int(xf))
    except:
        pass

    return s

# ============================================================
# Helper: build transfer edges among stops within distance threshold
# ============================================================
def build_transfer_edges_by_distance(
    stops_gdf,
    thresh_m=300,
    v_walk_kmh=4.0,
    make_bidir=True,
    edge_type="transfer_rail",
    mode="walk_transfer"
):
    """
    stops_gdf must be projected CRS in meters and contain:
      - node_id (string)
      - geometry (Point)

    Returns:
      GeoDataFrame with columns:
      [u, v, edge_type, mode, length, time, geometry]
    """
    if stops_gdf is None or len(stops_gdf) == 0:
        return gpd.GeoDataFrame(
            columns=["u", "v", "edge_type", "mode", "length", "time", "geometry"],
            crs=stops_gdf.crs if stops_gdf is not None else None
        )

    st = stops_gdf.copy()
    st = st[st.geometry.notna() & (st.geometry.geom_type == "Point")].copy()
    st["node_id"] = st["node_id"].astype(str)
    st = st.reset_index(drop=True)
    st["_sid"] = st.index.astype(int)

    sidx = st.sindex
    edges = []
    seen = set()

    for i, row in st.iterrows():
        p = row.geometry
        sid_i = int(row["_sid"])
        u = row["node_id"]

        bbox = p.buffer(thresh_m).bounds
        cand_idx = list(sidx.intersection(bbox))

        for j in cand_idx:
            if j == i:
                continue

            sid_j = int(st.at[j, "_sid"])
            if sid_j <= sid_i:
                continue

            key = (sid_i, sid_j)
            if key in seen:
                continue

            q = st.at[j, "geometry"]
            d = float(p.distance(q))

            if d <= thresh_m:
                v = str(st.at[j, "node_id"])
                t = d / 1000.0 / float(v_walk_kmh) * 60.0

                edges.append({
                    "u": u,
                    "v": v,
                    "edge_type": edge_type,
                    "mode": mode,
                    "length": d / 1000.0,
                    "time": t,
                    "geometry": LineString([p, q])
                })

                if make_bidir:
                    edges.append({
                        "u": v,
                        "v": u,
                        "edge_type": edge_type,
                        "mode": mode,
                        "length": d / 1000.0,
                        "time": t,
                        "geometry": LineString([q, p])
                    })

                seen.add(key)

    return gpd.GeoDataFrame(edges, crs=stops_gdf.crs)


# ============================================================
# Helper: extract stop points from PT edges
# ============================================================
def stops_from_pt_edges(edges_gdf, mode):
    """
    Robust version:
    1) collect ALL unique stop_ids from from_stop / to_stop
    2) try to infer representative geometry from line endpoints
    3) ensure every stop_id referenced by edges gets a node_id
    """
    g = edges_gdf.copy()
    g = g[g.geometry.notna()].copy()
    g = g[g.geometry.geom_type == "LineString"].copy()
    g = g[g["from_stop"].notna() & g["to_stop"].notna()].copy()

    if len(g) == 0:
        return gpd.GeoDataFrame(
            columns=["mode", "stop_id", "node_id", "node_type", "geometry"],
            crs=edges_gdf.crs
        )

    all_stop_ids = pd.unique(
        pd.concat([
            g["from_stop"].map(normalize_stop_id),
            g["to_stop"].map(normalize_stop_id)
        ], ignore_index=True)
    )

    all_ids_df = pd.DataFrame({
        "mode": mode,
        "stop_id": all_stop_ids
    })

    rows = []
    for _, r in g.iterrows():
        coords = list(r.geometry.coords)
        if len(coords) < 2:
            continue

        rows.append({
            "mode": mode,
            "stop_id": normalize_stop_id(r["from_stop"]),
            "x": coords[0][0],
            "y": coords[0][1]
        })
        rows.append({
            "mode": mode,
            "stop_id": normalize_stop_id(r["to_stop"]),
            "x": coords[-1][0],
            "y": coords[-1][1]
        })

    if len(rows) > 0:
        pts = pd.DataFrame(rows)
        agg_xy = pts.groupby(["mode", "stop_id"])[["x", "y"]].mean().reset_index()
        out = all_ids_df.merge(agg_xy, on=["mode", "stop_id"], how="left")
    else:
        out = all_ids_df.copy()
        out["x"] = np.nan
        out["y"] = np.nan

    out["geometry"] = [
        Point(x, y) if pd.notna(x) and pd.notna(y) else None
        for x, y in zip(out["x"], out["y"])
    ]

    stops = gpd.GeoDataFrame(
        out[["mode", "stop_id", "geometry"]],
        crs=g.crs
    )
    
    stops["node_id"] = stops.apply(
        lambda r: f"stop:{r['mode']}:{r['stop_id']}", axis=1
        )
    stops["node_type"] = "stop"

    return stops


# ============================================================
# Build PT nodes and edges
# ============================================================
def build_pt_gdfs(
    edges_by_mode,
    target_epsg,
    V_MODE,
    transfer_thresh_m=150.0,
    transfer_walk_kmh=4.0,
    transfer_merge_modes=("subway", "light_rail"),
):
    """
    edges_by_mode: dict {mode: edges_gdf_in_4326}
      each edges_gdf must contain:
      - from_stop
      - to_stop
      - geometry(LineString)

    target_epsg: projected CRS for meters
    V_MODE: dict {mode: speed_kmh}

    Returns:
      stop_nodes      : all PT stop nodes
      ride_edges_all  : ride edges + transfer edges
      stops_all       : raw stops gdf
      transfer_edges  : transfer-only edges
    """
    stop_gdfs = []
    ride_edges_list = []

    for mode, e_ll in edges_by_mode.items():
        if e_ll is None or len(e_ll) == 0:
            continue

        if e_ll.crs is None:
            e_ll = e_ll.set_crs("EPSG:4326")

        e = e_ll.to_crs(epsg=target_epsg).copy()
        e = e[e.geometry.notna()]
        e = e[e.geometry.geom_type == "LineString"].copy()
        e = e[e.geometry.apply(lambda g: len(list(g.coords)) >= 2)].copy()

        if len(e) == 0:
            continue

        e["length"] = e.geometry.length.astype(float) / 1000.0

        if mode not in V_MODE:
            raise ValueError(f"V_MODE missing speed for mode='{mode}'")

        e["time"] = e["length"] / float(V_MODE[mode]) * 60.0

        # Extract stops using the same e
        stops = stops_from_pt_edges(e, mode)

        stop_gdfs.append(stops)

        ride_edges = gpd.GeoDataFrame({
            "u": e["from_stop"].map(lambda x: f"stop:{mode}:{normalize_stop_id(x)}"),
            "v": e["to_stop"].map(lambda x: f"stop:{mode}:{normalize_stop_id(x)}"),
            "edge_type": f"ride_{mode}",
            "mode": mode,
            "length": e["length"].astype(float),
            "time": e["time"].astype(float),
            "geometry": e.geometry
        }, crs=e.crs)

        ride_edges_list.append(ride_edges)

    if stop_gdfs:
        stops_all = gpd.GeoDataFrame(
            pd.concat(stop_gdfs, ignore_index=True),
            crs=f"EPSG:{target_epsg}"
        )
    else:
        stops_all = gpd.GeoDataFrame(
            columns=["mode", "stop_id", "node_id", "node_type", "geometry"],
            crs=f"EPSG:{target_epsg}"
        )

    if ride_edges_list:
        ride_edges_all = gpd.GeoDataFrame(
            pd.concat(ride_edges_list, ignore_index=True),
            crs=f"EPSG:{target_epsg}"
        )
    else:
        ride_edges_all = gpd.GeoDataFrame(
            columns=["u", "v", "edge_type", "mode", "length", "time", "geometry"],
            crs=f"EPSG:{target_epsg}"
        )

    if len(stops_all) > 0:
        stop_nodes = stops_all.copy()
        stop_nodes["node_type"] = "stop"
    else:
        stop_nodes = gpd.GeoDataFrame(
            columns=["mode", "stop_id", "node_id", "node_type", "geometry"],
            crs=f"EPSG:{target_epsg}"
        )

    transfer_edges = gpd.GeoDataFrame(
        columns=["u", "v", "edge_type", "mode", "length", "time", "geometry"],
        crs=f"EPSG:{target_epsg}"
    )

    if len(stops_all) > 0:
        need_cols = {"mode", "node_id", "geometry"}
        if not need_cols.issubset(set(stops_all.columns)):
            raise ValueError(f"stops_all missing columns: {need_cols - set(stops_all.columns)}")

        rail_stops = stops_all[stops_all["mode"].isin(list(transfer_merge_modes))].copy()

        if len(rail_stops) > 1:
            transfer_edges = build_transfer_edges_by_distance(
                rail_stops,
                thresh_m=float(transfer_thresh_m),
                v_walk_kmh=float(transfer_walk_kmh),
                make_bidir=True,
                edge_type="transfer_rail",
                mode="walk_transfer"
            )

            ride_edges_all = gpd.GeoDataFrame(
                pd.concat([ride_edges_all, transfer_edges], ignore_index=True),
                crs=f"EPSG:{target_epsg}"
            )

    return stop_nodes, ride_edges_all, stops_all, transfer_edges


# ============================================================
# Build walk nodes and edges
# ============================================================
def build_walk_gdfs_from_polygon(boundary, target_epsg, V_WALK=4.0):
    """
    boundary: GeoDataFrame with polygon geometry
    target_epsg: projected CRS in meters
    """
    poly = boundary.to_crs("EPSG:4326").union_all()
    G = ox.graph_from_polygon(
        poly,
        network_type="walk",
        simplify=True,
        truncate_by_edge=True
    )

    wn = ox.graph_to_gdfs(G, edges=False)
    walk_nodes = gpd.GeoDataFrame(
        wn,
        geometry=gpd.points_from_xy(wn["x"], wn["y"]),
        crs="EPSG:4326"
    )
    walk_nodes["node_id"] = walk_nodes.index.astype(str)
    walk_nodes["node_type"] = "walk"
    walk_nodes = walk_nodes.to_crs(epsg=target_epsg)

    we = ox.graph_to_gdfs(G, nodes=False).reset_index()
    walk_edges = gpd.GeoDataFrame(we, geometry="geometry", crs="EPSG:4326").to_crs(epsg=target_epsg)

    walk_edges["length"] = walk_edges.geometry.length.astype(float) / 1000.0
    walk_edges["time"] = walk_edges["length"] / float(V_WALK) * 60.0

    walk_edges_std = gpd.GeoDataFrame({
        "u": walk_edges["u"].astype(str),
        "v": walk_edges["v"].astype(str),
        "edge_type": "walk",
        "mode": "walk",
        "length": walk_edges["length"].astype(float),
        "time": walk_edges["time"].astype(float),
        "geometry": walk_edges.geometry
    }, crs=walk_edges.crs)

    return walk_nodes, walk_edges_std


# ============================================================
# K nearest walk nodes by expanding spatial search
# ============================================================
def k_nearest_walk_nodes_by_expanding_search(
    point,
    wn,
    sidx,
    k=4,
    r0=200,
    rmax=5000,
    step=2.0
):
    """
    point / wn must be in projected CRS (meters).
    GeoPandas 1.1.1 compatible via sindex.intersection.
    """
    r = r0
    while r <= rmax:
        bbox = point.buffer(r).bounds
        cand_idx = list(sidx.intersection(bbox))

        if len(cand_idx) >= k:
            cands = wn.iloc[cand_idx]
            d = cands.geometry.distance(point).values
            order = np.argsort(d)[:k]
            return cands.iloc[order].copy(), d[order]

        r *= step

    d_all = wn.geometry.distance(point).values
    order = np.argsort(d_all)[:k]
    return wn.iloc[order].copy(), d_all[order]


# ============================================================
# Connect PT stops to nearest k walk nodes
# ============================================================
def attach_stops_to_walk_nodes_knearest(
    stops_gdf,
    walk_nodes,
    target_epsg,
    k=2,
    v_walk_kmh=4.0,
    r0=200,
    rmax=5000,
    step=2.0,
    max_dist_m=None,
    make_bidir=True
):
    """
    Each transit stop connects to nearest k walk nodes.

    Returns:
      stop_access_edges
    """
    if stops_gdf.crs is None:
        stops_gdf = stops_gdf.set_crs("EPSG:4326")
    if walk_nodes.crs is None:
        walk_nodes = walk_nodes.set_crs("EPSG:4326")

    st = stops_gdf.to_crs(epsg=target_epsg).copy()
    wn = walk_nodes.to_crs(epsg=target_epsg).copy()

    for col in ["node_id", "geometry"]:
        if col not in st.columns:
            raise ValueError(f"stops_gdf missing column: {col}")
        if col not in wn.columns:
            raise ValueError(f"walk_nodes missing column: {col}")

    st = st[st.geometry.notna() & (st.geometry.geom_type == "Point")].copy()
    wn = wn[wn.geometry.notna() & (wn.geometry.geom_type == "Point")].copy()

    st["node_id"] = st["node_id"].astype(str)
    wn["node_id"] = wn["node_id"].astype(str)

    sidx = wn.sindex
    edges = []
    seen = set()

    for _, row in tqdm(st.iterrows(), total=len(st), desc="Attach stops to walk nodes"):
        stop_id = str(row["node_id"])
        p = row.geometry

        nbrs, dists = k_nearest_walk_nodes_by_expanding_search(
            p, wn, sidx, k=k, r0=r0, rmax=rmax, step=step
        )

        for nid, dist, q in zip(nbrs["node_id"].values, dists, nbrs.geometry.values):
            nid = str(nid)
            dist_m = float(dist)

            if max_dist_m is not None and dist_m > float(max_dist_m):
                continue

            dist_km = dist_m / 1000.0
            time_min = dist_km / float(v_walk_kmh) * 60.0

            key1 = (stop_id, nid)
            if key1 not in seen:
                edges.append({
                    "u": stop_id,
                    "v": nid,
                    "edge_type": "stop_connector",
                    "mode": "walk",
                    "length": dist_km,
                    "time": time_min,
                    "geometry": LineString([p, q])
                })
                seen.add(key1)

            if make_bidir:
                key2 = (nid, stop_id)
                if key2 not in seen:
                    edges.append({
                        "u": nid,
                        "v": stop_id,
                        "edge_type": "stop_connector",
                        "mode": "walk",
                        "length": dist_km,
                        "time": time_min,
                        "geometry": LineString([q, p])
                    })
                    seen.add(key2)

    return gpd.GeoDataFrame(edges, crs=f"EPSG:{target_epsg}")


# ============================================================
# Connect zone centroids to nearest k walk nodes
# ============================================================
def attach_centroids_to_walk_nodes(
    centroids_gdf,
    walk_nodes,
    target_epsg,
    k=4,
    v_walk_kmh=4.0,
    r0=200,
    rmax=5000,
    step=2.0,
    max_dist_m=None,
    make_bidir=True
):
    """
    centroids_gdf must contain:
      - zone_id
      - geometry(Point)

    Returns:
      centroid_nodes
      centroid_access_edges
    """
    if centroids_gdf.crs is None:
        centroids_gdf = centroids_gdf.set_crs("EPSG:4326")
    if walk_nodes.crs is None:
        walk_nodes = walk_nodes.set_crs("EPSG:4326")

    cz = centroids_gdf.to_crs(epsg=target_epsg).copy()
    wn = walk_nodes.to_crs(epsg=target_epsg).copy()

    if "zone_id" not in cz.columns:
        raise ValueError("centroids_gdf 需要 zone_id 列")
    if "node_id" not in wn.columns:
        raise ValueError("walk_nodes 需要 node_id 列")

    cz = cz[cz.geometry.notna() & (cz.geometry.geom_type == "Point")].copy()
    wn = wn[wn.geometry.notna() & (wn.geometry.geom_type == "Point")].copy()

    cz["zone_id"] = cz["zone_id"].astype(str)
    wn["node_id"] = wn["node_id"].astype(str)

    sidx = wn.sindex

    centroid_nodes = cz[["zone_id", "geometry"]].copy()
    centroid_nodes["node_id"] = centroid_nodes["zone_id"].map(lambda z: f"zone:{z}")
    centroid_nodes["node_type"] = "zone_centroid"
    centroid_nodes = gpd.GeoDataFrame(centroid_nodes, crs=f"EPSG:{target_epsg}")

    edges = []

    for row in tqdm(centroid_nodes.itertuples(), total=len(centroid_nodes), desc="Attach centroids to walk nodes"):
        z_node = row.node_id
        p = row.geometry

        nbrs, dists = k_nearest_walk_nodes_by_expanding_search(
            p, wn, sidx, k=k, r0=r0, rmax=rmax, step=step
        )

        for nid, dist, q in zip(nbrs["node_id"].values, dists, nbrs.geometry.values):
            dist_m = float(dist)

            if max_dist_m is not None and dist_m > float(max_dist_m):
                continue

            dist_km = dist_m / 1000.0
            time_min = dist_km / float(v_walk_kmh) * 60.0

            edges.append({
                "u": z_node,
                "v": str(nid),
                "edge_type": "zone_connector",
                "mode": "walk",
                "length": dist_km,
                "time": time_min,
                "geometry": LineString([p, q])
            })

            if make_bidir:
                edges.append({
                    "u": str(nid),
                    "v": z_node,
                    "edge_type": "zone_connector",
                    "mode": "walk",
                    "length": dist_km,
                    "time": time_min,
                    "geometry": LineString([q, p])
                })

    centroid_access_edges = gpd.GeoDataFrame(edges, crs=f"EPSG:{target_epsg}")
    return centroid_nodes, centroid_access_edges


# ============================================================
# Main pipeline: build multimodal network
# ============================================================
def build_multimodal_network(
    boundary,
    centroids_gdf,
    edges_by_mode,
    target_epsg,
    V_MODE,
    V_WALK=4.0,
    stop_k=2,
    centroid_k=4,
    transfer_thresh_m=150.0,
    transfer_merge_modes=("subway", "light_rail"),
    stop_r0=200,
    stop_rmax=5000,
    stop_step=2.0,
    stop_max_dist_m=None,
    centroid_r0=200,
    centroid_rmax=5000,
    centroid_step=2.0,
    centroid_max_dist_m=None
):
    """
    Final simplified multimodal network builder.

    Returns a dict:
      {
        "all_nodes",
        "all_edges",
        "walk_nodes",
        "walk_edges",
        "stop_nodes",
        "ride_edges",
        "transfer_edges",
        "stop_access_edges",
        "centroid_nodes",
        "centroid_access_edges",
        "stops_all"
      }
    """

    # 1) PT
    stop_nodes, ride_edges_all, stops_all, transfer_edges = build_pt_gdfs(
        edges_by_mode=edges_by_mode,
        target_epsg=target_epsg,
        V_MODE=V_MODE,
        transfer_thresh_m=transfer_thresh_m,
        transfer_walk_kmh=V_WALK,
        transfer_merge_modes=transfer_merge_modes
    )

    # 2) walk
    walk_nodes, walk_edges = build_walk_gdfs_from_polygon(
        boundary=boundary,
        target_epsg=target_epsg,
        V_WALK=V_WALK
    )

    # 3) stops -> nearest 2 walk nodes
    stop_access_edges = attach_stops_to_walk_nodes_knearest(
        stops_gdf=stop_nodes,
        walk_nodes=walk_nodes,
        target_epsg=target_epsg,
        k=stop_k,
        v_walk_kmh=V_WALK,
        r0=stop_r0,
        rmax=stop_rmax,
        step=stop_step,
        max_dist_m=stop_max_dist_m,
        make_bidir=True
    )

    # 4) centroids -> nearest 4 walk nodes
    centroid_nodes, centroid_access_edges = attach_centroids_to_walk_nodes(
        centroids_gdf=centroids_gdf,
        walk_nodes=walk_nodes,
        target_epsg=target_epsg,
        k=centroid_k,
        v_walk_kmh=V_WALK,
        r0=centroid_r0,
        rmax=centroid_rmax,
        step=centroid_step,
        max_dist_m=centroid_max_dist_m,
        make_bidir=True
    )

    # 5) all nodes
    walk_nodes2 = walk_nodes.copy()
    if "node_type" not in walk_nodes2.columns:
        walk_nodes2["node_type"] = "walk"

    stop_nodes2 = stop_nodes.copy()
    if "node_type" not in stop_nodes2.columns:
        stop_nodes2["node_type"] = "stop"

    centroid_nodes2 = centroid_nodes.copy()
    if "node_type" not in centroid_nodes2.columns:
        centroid_nodes2["node_type"] = "zone_centroid"

    all_nodes = gpd.GeoDataFrame(
        pd.concat([
            walk_nodes2[["node_id", "node_type", "geometry"]],
            stop_nodes2[["node_id", "node_type", "geometry"]],
            centroid_nodes2[["node_id", "node_type", "geometry"]],
        ], ignore_index=True),
        crs=f"EPSG:{target_epsg}"
    )

    # Deduplication (by node_id)
    all_nodes = all_nodes.drop_duplicates(subset=["node_id"]).reset_index(drop=True)

    # 6) all edges
    all_edges = gpd.GeoDataFrame(
        pd.concat([
            walk_edges[["u", "v", "edge_type", "mode", "length", "time", "geometry"]],
            ride_edges_all[["u", "v", "edge_type", "mode", "length", "time", "geometry"]],
            stop_access_edges[["u", "v", "edge_type", "mode", "length", "time", "geometry"]],
            centroid_access_edges[["u", "v", "edge_type", "mode", "length", "time", "geometry"]],
        ], ignore_index=True),
        crs=f"EPSG:{target_epsg}"
    )

    return {
        "all_nodes": all_nodes,
        "all_edges": all_edges,
        "walk_nodes": walk_nodes,
        "walk_edges": walk_edges,
        "stop_nodes": stop_nodes,
        "ride_edges": ride_edges_all,
        "transfer_edges": transfer_edges,
        "stop_access_edges": stop_access_edges,
        "centroid_nodes": centroid_nodes,
        "centroid_access_edges": centroid_access_edges,
        "stops_all": stops_all
    }

### Run each city

In [ ]:
epsg_dct = {"beijing":4548,"shanghai":4549,"shenzhen":4547,"nanjing":4549,"london":27700,"losangeles":26911,"newyork":26918}

city = "london"
zone_name = "msoa"
TARGET_EPSG = epsg_dct[city]
boundary = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_boundary.shp").to_crs(epsg=TARGET_EPSG)
boundary['geometry'] = boundary.buffer(5000)
boundary = boundary.to_crs("EPSG:4326")
zone_centroid = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{zone_name}.shp")
zone_centroid['geometry'] = zone_centroid.centroid
zone_centroid['zone_id'] = zone_centroid['id']

train_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/train_edges.shp")
subway_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/subway_edges.shp")
light_rail_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/light_rail_edges.shp")
tram_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/tram_edges.shp")
bus_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/bus_edges.shp")

edges_by_mode = {
    "train": train_edges,
    "subway": subway_edges,
    "light_rail": light_rail_edges,
    "tram": tram_edges,
    "bus": bus_edges
    }
V_MODE={"train":60,"subway":40,"light_rail":40,"tram":25,"bus":20}

result = build_multimodal_network(
    boundary=boundary,
    centroids_gdf=zone_centroid,
    edges_by_mode=edges_by_mode,
    target_epsg=TARGET_EPSG,
    V_MODE=V_MODE,
    V_WALK=4.0,
    stop_k=4,
    centroid_k=4,
    transfer_thresh_m=150,
    transfer_merge_modes=("train","subway","light_rail"),
    stop_max_dist_m=None,
    centroid_max_dist_m=None
    )

all_edges = result["all_edges"]
all_edges.to_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/public_transport_edges.shp")
all_edges[['u','v','edge_type','mode','length','time']].to_csv(f"D:/urban_hierarchy_congestion/data/tranport_network/table_data/{city}/public_transport_edges.csv",index=False)

city = "losangeles"
zone_name = "tract"
TARGET_EPSG = epsg_dct[city]
boundary = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_boundary.shp").to_crs(epsg=TARGET_EPSG)
boundary['geometry'] = boundary.buffer(5000)
boundary = boundary.to_crs("EPSG:4326")
zone_centroid = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{zone_name}.shp")
zone_centroid['geometry'] = zone_centroid.centroid
zone_centroid['zone_id'] = zone_centroid['id']

train_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/train_edges.shp")
subway_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/subway_edges.shp")
light_rail_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/light_rail_edges.shp")
tram_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/tram_edges.shp")
bus_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/bus_edges.shp")

edges_by_mode = {
    "train": train_edges,
    "subway": subway_edges,
    "light_rail": light_rail_edges,
    "tram": tram_edges,
    "bus": bus_edges
    }
V_MODE={"train":60,"subway":40,"light_rail":40,"tram":25,"bus":20}

result = build_multimodal_network(
    boundary=boundary,
    centroids_gdf=zone_centroid,
    edges_by_mode=edges_by_mode,
    target_epsg=TARGET_EPSG,
    V_MODE=V_MODE,
    V_WALK=4.0,
    stop_k=4,
    centroid_k=4,
    transfer_thresh_m=150,
    transfer_merge_modes=("train","subway","light_rail"),
    stop_max_dist_m=None,
    centroid_max_dist_m=None
    )

all_edges = result["all_edges"]
all_edges.to_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/public_transport_edges.shp")
all_edges[['u','v','edge_type','mode','length','time']].to_csv(f"D:/urban_hierarchy_congestion/data/tranport_network/table_data/{city}/public_transport_edges.csv",index=False)

city = "newyork"
zone_name = "tract"
TARGET_EPSG = epsg_dct[city]
boundary = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_boundary.shp").to_crs(epsg=TARGET_EPSG)
boundary['geometry'] = boundary.buffer(5000)
boundary = boundary.to_crs("EPSG:4326")
zone_centroid = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{zone_name}.shp")
zone_centroid['geometry'] = zone_centroid.centroid
zone_centroid['zone_id'] = zone_centroid['id']

train_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/train_edges.shp")
subway_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/subway_edges.shp")
light_rail_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/light_rail_edges.shp")
bus_edges = gpd.read_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/bus_edges.shp")

edges_by_mode = {
    "train": train_edges,
    "subway": subway_edges,
    "light_rail": light_rail_edges,
    "bus": bus_edges
    }
V_MODE={"train":60,"subway":40,"light_rail":40,"tram":25,"bus":20}

result = build_multimodal_network(
    boundary=boundary,
    centroids_gdf=zone_centroid,
    edges_by_mode=edges_by_mode,
    target_epsg=TARGET_EPSG,
    V_MODE=V_MODE,
    V_WALK=4.0,
    stop_k=4,
    centroid_k=4,
    transfer_thresh_m=150,
    transfer_merge_modes=("train","subway","light_rail"),
    stop_max_dist_m=None,
    centroid_max_dist_m=None
    )

all_edges = result["all_edges"]
all_edges.to_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/public_transport_edges.shp")
all_edges[['u','v','edge_type','mode','length','time']].to_csv(f"D:/urban_hierarchy_congestion/data/tranport_network/table_data/{city}/public_transport_edges.csv",index=False)

for city in ["beijing","shanghai","shenzhen"]:
    zone_name = "grid_1k"
    TARGET_EPSG = epsg_dct[city]
    boundary = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_boundary.shp").to_crs(epsg=TARGET_EPSG)
    boundary['geometry'] = boundary.buffer(5000)
    boundary = boundary.to_crs("EPSG:4326")
    zone_centroid = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{zone_name}.shp")
    zone_centroid['geometry'] = zone_centroid.centroid
    zone_centroid['zone_id'] = zone_centroid['id']


    subway_edges = gpd.read_file(f"tranport network/spatial data/{city}/metro_edges.shp")
    subway_edges.rename(columns={'s_stopid':'from_stop','e_stopid':'to_stop'},inplace=True)
    bus_edges = gpd.read_file(f"tranport network/spatial data/{city}/bus_edges.shp")
    bus_edges.rename(columns={'s_stopid':'from_stop','e_stopid':'to_stop'},inplace=True)

    edges_by_mode = {
        "subway": subway_edges,
        "bus": bus_edges
        }
    V_MODE={"subway":40,"bus":20}

    result = build_multimodal_network(
        boundary=boundary,
        centroids_gdf=zone_centroid,
        edges_by_mode=edges_by_mode,
        target_epsg=TARGET_EPSG,
        V_MODE=V_MODE,
        V_WALK=4.0,
        stop_k=4,
        centroid_k=4,
        transfer_thresh_m=0,
        transfer_merge_modes=("subway"),
        stop_max_dist_m=None,
        centroid_max_dist_m=None
        )

    all_edges = result["all_edges"]
    all_edges.to_file(f"D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/public_transport_edges.shp")
    all_edges[['u','v','edge_type','mode','length','time']].to_csv(f"D:/urban_hierarchy_congestion/data/tranport_network/table_data/{city}/public_transport_edges.csv",index=False)

# Build drive network

In [ ]:
def build_drive_edges(unit_path,crs_id,
                      edges_shp_output_path,edges_csv_output_path):
    
    ox.settings.log_console=True
    ox.settings.use_cache=True
    unit = gpd.read_file(unit_path)
    unit.to_crs(crs=crs_id,inplace=True)
    unit_centroid = unit.copy()
    unit_centroid.geometry = unit_centroid.centroid
    unit_centroid['zone_id'] = unit_centroid['id'].astype(str)
    unit_centroid["node_id"] = unit_centroid["zone_id"].map(lambda z: f"zone:{z}")
    unit_centroid.drop(columns={'zone_id'},inplace=True)
    unit_centroid['x'] = unit_centroid.geometry.x
    unit_centroid['y'] = unit_centroid.geometry.y
    
    boundary = gpd.GeoDataFrame(geometry=[unit.union_all()], crs=unit.crs)
    boundary['geometry'] = boundary.buffer(5000)
    boundary.to_crs("EPSG:4326",inplace=True)
    G = ox.graph_from_polygon(boundary['geometry'].iloc[0], network_type='drive', truncate_by_edge=True)
    
    default_lanes = {"motorway":3,"motorway_link":1,"trunk":3,"trunk_link":1,"primary":2,"primary_link":1,
                     "secondary":2,"secondary_link":1,"tertiary":1,"tertiary_link":1,"unclassified":1,"residential":1}

    default_speed = {"motorway":100,"motorway_link":60,"trunk":80,"trunk_link":50,"primary":60,"primary_link":40,
                     "secondary":50,"secondary_link":35,"tertiary":40,"tertiary_link":30,"unclassified":30,"residential":25}

    capacity_per_lane = {"motorway":2000,"motorway_link":1800,"trunk":2000,"trunk_link":1800,"primary":1800,
                         "primary_link":1800,"secondary":1800,"secondary_link":1800,"tertiary":1600,
                         "tertiary_link":1600,"unclassified":1500,"residential":1200}

    def parse_lanes(x):
        if x is None:
            return np.nan
        
        if isinstance(x, (list, tuple, np.ndarray)):
            x = x[0]
        
        try:
            return int(str(x).split(";")[0])
        except:
            return np.nan

    def parse_speed(x):

        if x is None:
            return np.nan

        if isinstance(x, (list, tuple, np.ndarray)):
            x = x[0]

        s = str(x).lower()

        # Extract the number
        match = re.search(r'\d+', s)
        if match is None:
            return np.nan

        speed = float(match.group())

        # mph to km/h
        if "mph" in s:
            speed *= 1.60934

        return speed
        
    nodes, edges = ox.graph_to_gdfs(G)

    nodes.reset_index(drop=False,inplace=True)
    nodes.rename(columns={'osmid':'node_id'},inplace=True)
    nodes["node_id"] = nodes["node_id"].astype(str)
    nodes.to_crs(crs=crs_id,inplace=True)
    nodes['x'] = nodes.geometry.x
    nodes['y'] = nodes.geometry.y
    nodes = nodes[['node_id','x','y','geometry']].copy()

    edges.reset_index(drop=False,inplace=True)
    edges['duplicated'] = edges.duplicated(subset=["u", "v"])
    edges = edges.loc[(edges['duplicated']==False)].copy()
    edges.index = range(len(edges))
    edges['u'] = edges['u'].astype(str)
    edges['v'] = edges['v'].astype(str)
    edges.to_crs(crs=crs_id,inplace=True)
    edges['length'] = edges.geometry.length/1000
    edges = edges.loc[(edges['length']>0)].copy()
    edges.index = range(len(edges))

    edges["highway"] = edges["highway"].apply(lambda x: x[0] if isinstance(x,list) else x)
    edges["lanes_clean"] = edges["lanes"].apply(parse_lanes)
    edges["speed_clean"] = edges["maxspeed"].apply(parse_speed)
    edges["lanes_final"] = edges.apply(
        lambda row: row["lanes_clean"] 
        if not pd.isna(row["lanes_clean"]) and row["lanes_clean"]>0
        else default_lanes.get(row["highway"],1),
        axis=1
    )
    edges["capacity"] = edges.apply(
        lambda row: row["lanes_final"]*capacity_per_lane.get(row["highway"],1200),
        axis=1
    )
    edges["speed_final"] = edges.apply(
        lambda row: row["speed_clean"] 
        if not pd.isna(row["speed_clean"]) and row["speed_clean"]>0
        else default_speed.get(row["highway"],30),
        axis=1
    )
    edges['fft'] = 1.3*edges['length']/edges['speed_final']*60
    edges['alpha'] = 0.6
    edges['beta'] = 4
    edges = edges[['u','v','highway','length','capacity','fft','alpha','beta','geometry']].copy()

    # Record all connecting cables
    new_connections = []

    for i, row in unit_centroid.iterrows():
        point = row.geometry
        nodes['distance'] = nodes.geometry.distance(point)
        nearest_4 = nodes.nsmallest(4, 'distance').copy()
        
        # Draw a line connecting each pair of nearest points
        for _, node in nearest_4.iterrows():
            line_geom = geometry.LineString([point, node.geometry])
            connection = {
                'geometry': line_geom,
                'u': row.node_id,   
                'v': node.node_id       
            }
            new_connections.append(connection)
            
            line_geom = geometry.LineString([node.geometry,point])
            connection = {
                'geometry': line_geom,  
                'u': node.node_id,   
                'v': row.node_id       
            }
            new_connections.append(connection)

    connection_gdf = gpd.GeoDataFrame(new_connections, crs=crs_id)

    connection_gdf['highway'] = 'connector'
    connection_gdf['length'] = connection_gdf.geometry.length/1000
    connection_gdf['fft'] = 1.3*connection_gdf['length']/20*60
    connection_gdf['capacity'] = 1e6
    connection_gdf['alpha'] = 0.6
    connection_gdf['beta'] = 4

    edges_new = gpd.GeoDataFrame(pd.concat([edges, connection_gdf[edges.columns]], ignore_index=True), crs=edges.crs)
    edges_new['u'] = edges_new['u'].astype(str)
    edges_new['v'] = edges_new['v'].astype(str)
    edges_new.to_file(edges_shp_output_path,encoding='utf-8')

    edges_new[['u','v','highway','length','capacity','fft','alpha','beta']].to_csv(edges_csv_output_path,index=False)

In [ ]:
city_params = {'beijing':{'crs_id':'EPSG:4548', 'unit_name':'grid_1k'},
               'shanghai':{'crs_id':'EPSG:4549', 'unit_name':'grid_1k'},
               'shenzhen':{'crs_id':'EPSG:4547', 'unit_name':'grid_1k'},
               'nanjing':{'crs_id':'EPSG:4549', 'unit_name':'grid_1k'},
               'london':{'crs_id':'EPSG:27700', 'unit_name':'msoa'},
               'losangeles':{'crs_id':'EPSG:26911', 'unit_name':'tract'},
               'newyork':{'crs_id':'EPSG:26918', 'unit_name':'tract'}}

In [ ]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    build_drive_edges(f'D:/urban_hierarchy_congestion/data/taz/{city}_{city_params[city]['unit_name']}.shp',city_params[city]['crs_id'],
                      f'D:/urban_hierarchy_congestion/data/tranport_network/spatial_data/{city}/drive_edges.shp',f'D:/urban_hierarchy_congestion/data/tranport_network/table_data/{city}/drive_edges.csv')

# Build walk network

In [ ]:
def build_walk_edges(unit_path,crs_id,
                      edges_shp_output_path,edges_csv_output_path):
    
    ox.settings.log_console=True
    ox.settings.use_cache=True
    unit = gpd.read_file(unit_path)
    unit.to_crs(crs=crs_id,inplace=True)
    unit_centroid = unit.copy()
    unit_centroid.geometry = unit_centroid.centroid
    unit_centroid['zone_id'] = unit_centroid['id'].astype(str)
    unit_centroid["node_id"] = unit_centroid["zone_id"].map(lambda z: f"zone:{z}")
    unit_centroid.drop(columns={'zone_id'},inplace=True)
    unit_centroid['x'] = unit_centroid.geometry.x
    unit_centroid['y'] = unit_centroid.geometry.y
    
    boundary = gpd.GeoDataFrame(geometry=[unit.union_all()], crs=unit.crs)
    boundary['geometry'] = boundary.buffer(5000)
    boundary.to_crs("EPSG:4326",inplace=True)
    G = ox.graph_from_polygon(boundary['geometry'].iloc[0], network_type='walk', truncate_by_edge=True)
        
    nodes, edges = ox.graph_to_gdfs(G)

    nodes.reset_index(drop=False,inplace=True)
    nodes.rename(columns={'osmid':'node_id'},inplace=True)
    nodes["node_id"] = nodes["node_id"].astype(str)
    nodes.to_crs(crs=crs_id,inplace=True)
    nodes['x'] = nodes.geometry.x
    nodes['y'] = nodes.geometry.y
    nodes = nodes[['node_id','x','y','geometry']].copy()

    edges.reset_index(drop=False,inplace=True)
    edges['duplicated'] = edges.duplicated(subset=["u", "v"])
    edges = edges.loc[(edges['duplicated']==False)].copy()
    edges.index = range(len(edges))
    edges['u'] = edges['u'].astype(str)
    edges['v'] = edges['v'].astype(str)
    edges.to_crs(crs=crs_id,inplace=True)
    edges['length'] = edges.geometry.length/1000
    edges = edges.loc[(edges['length']>0)].copy()
    edges.index = range(len(edges))
    edges['time'] = edges['length']/4*60

    edges = edges[['u','v','highway','length','time','geometry']].copy()

    # Record all connecting lines
    new_connections = []

    for i, row in unit_centroid.iterrows():
        point = row.geometry
        nodes['distance'] = nodes.geometry.distance(point)
        nearest_4 = nodes.nsmallest(4, 'distance').copy()
        
        # Draw lines connecting each pair of nearest points
        for _, node in nearest_4.iterrows():
            line_geom = geometry.LineString([point, node.geometry])
            connection = {
                'geometry': line_geom,
                'u': row.node_id,   
                'v': node.node_id       
            }
            new_connections.append(connection)
            
            line_geom = geometry.LineString([node.geometry,point])
            connection = {
                'geometry': line_geom,  
                'u': node.node_id,   
                'v': row.node_id       
            }
            new_connections.append(connection)

    connection_gdf = gpd.GeoDataFrame(new_connections, crs=crs_id)

    connection_gdf['highway'] = 'connector'
    connection_gdf['length'] = connection_gdf.geometry.length/1000
    connection_gdf['time'] = connection_gdf['length']/4*60

    edges_new = gpd.GeoDataFrame(pd.concat([edges, connection_gdf[edges.columns]], ignore_index=True), crs=edges.crs)
    edges_new['u'] = edges_new['u'].astype(str)
    edges_new['v'] = edges_new['v'].astype(str)
    edges_new.to_file(edges_shp_output_path,encoding='utf-8')

    edges_new[['u','v','highway','length','time']].to_csv(edges_csv_output_path,index=False)

In [ ]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    build_walk_edges(f'D:/urban_hierarchy_congestion/data/taz/{city}_{city_params[city]['unit_name']}.shp',city_params[city]['crs_id'],
                      f'transport network/spatial data/{city}/walk_edges.shp',f'transport network/table data/{city}/walk_edges.csv')

# Build cycle network

In [ ]:
def build_bike_edges(unit_path,crs_id,
                      edges_shp_output_path,edges_csv_output_path):
    
    ox.settings.log_console=True
    ox.settings.use_cache=True
    unit = gpd.read_file(unit_path)
    unit.to_crs(crs=crs_id,inplace=True)
    unit_centroid = unit.copy()
    unit_centroid.geometry = unit_centroid.centroid
    unit_centroid['zone_id'] = unit_centroid['id'].astype(str)
    unit_centroid["node_id"] = unit_centroid["zone_id"].map(lambda z: f"zone:{z}")
    unit_centroid.drop(columns={'zone_id'},inplace=True)
    unit_centroid['x'] = unit_centroid.geometry.x
    unit_centroid['y'] = unit_centroid.geometry.y
    
    boundary = gpd.GeoDataFrame(geometry=[unit.union_all()], crs=unit.crs)
    boundary['geometry'] = boundary.buffer(5000)
    boundary.to_crs("EPSG:4326",inplace=True)
    G = ox.graph_from_polygon(boundary['geometry'].iloc[0], network_type='bike', truncate_by_edge=True)
        
    nodes, edges = ox.graph_to_gdfs(G)

    nodes.reset_index(drop=False,inplace=True)
    nodes.rename(columns={'osmid':'node_id'},inplace=True)
    nodes["node_id"] = nodes["node_id"].astype(str)
    nodes.to_crs(crs=crs_id,inplace=True)
    nodes['x'] = nodes.geometry.x
    nodes['y'] = nodes.geometry.y
    nodes = nodes[['node_id','x','y','geometry']].copy()

    edges.reset_index(drop=False,inplace=True)
    edges['duplicated'] = edges.duplicated(subset=["u", "v"])
    edges = edges.loc[(edges['duplicated']==False)].copy()
    edges.index = range(len(edges))
    edges['u'] = edges['u'].astype(str)
    edges['v'] = edges['v'].astype(str)
    edges.to_crs(crs=crs_id,inplace=True)
    edges['length'] = edges.geometry.length/1000
    edges = edges.loc[(edges['length']>0)].copy()
    edges.index = range(len(edges))
    edges['time'] = edges['length']/15*60

    edges = edges[['u','v','highway','length','time','geometry']].copy()

    # Record all connecting lines
    new_connections = []

    for i, row in unit_centroid.iterrows():
        point = row.geometry
        nodes['distance'] = nodes.geometry.distance(point)
        nearest_4 = nodes.nsmallest(4, 'distance').copy()
        
        # Draw lines connecting each pair of nearest points
        for _, node in nearest_4.iterrows():
            line_geom = geometry.LineString([point, node.geometry])
            connection = {
                'geometry': line_geom,
                'u': row.node_id,   
                'v': node.node_id       
            }
            new_connections.append(connection)
            
            line_geom = geometry.LineString([node.geometry,point])
            connection = {
                'geometry': line_geom,  
                'u': node.node_id,   
                'v': row.node_id       
            }
            new_connections.append(connection)

    connection_gdf = gpd.GeoDataFrame(new_connections, crs=crs_id)

    connection_gdf['highway'] = 'connector'
    connection_gdf['length'] = connection_gdf.geometry.length/1000
    connection_gdf['time'] = connection_gdf['length']/15*60

    edges_new = gpd.GeoDataFrame(pd.concat([edges, connection_gdf[edges.columns]], ignore_index=True), crs=edges.crs)
    edges_new['u'] = edges_new['u'].astype(str)
    edges_new['v'] = edges_new['v'].astype(str)
    edges_new.to_file(edges_shp_output_path,encoding='utf-8')

    edges_new[['u','v','highway','length','time']].to_csv(edges_csv_output_path,index=False)

In [ ]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    build_bike_edges(f'D:/urban_hierarchy_congestion/data/taz/{city}_{city_params[city]['unit_name']}.shp',city_params[city]['crs_id'],
                      f'transport network/spatial data/{city}/bike_edges.shp',f'transport network/table data/{city}/bike_edges.csv')